## Import

In [1]:
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

## Dataset

In [3]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = '3b'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.4, -0.1, 0, 0.1, 0.4], rhos=[0.5, 0.6, 0.7, 0.8, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.77351285222025


## Hyperaparams

In [4]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## Joint training

In [6]:
## Vector copula estimation
from estimators.VCE_joint import VCE

hyperparams.K_components = 5
hyperparams.nde_type = 'VGC'

ret = []
time_spend = []
for i in range(8):
    t0 = time.time()
    
    estimator = VCE(None, None, architecture_critic, hyperparams)
    estimator.to(device)
    estimator.learn(X, Y)

    mi = estimator.MI(X, Y)
    t = time.time()-t0

    print('true MI:', dataset.empirical_mutual_info())
    print('est MI:', estimator.MI(X, Y))
    
    ret.append(mi)
    time_spend.append(t)

K components 5 joint learning True 

finished: t= 0 loss= 31698.953125 loss val= 31221.798828125 best val loss= 31221.798828125 best t= 0
finished: t= 101 loss= 152.30909729003906 loss val= 153.24095153808594 best val loss= 152.16192626953125 best t= 99
finished: t= 202 loss= 126.15973663330078 loss val= 129.7056884765625 best val loss= 128.84605407714844 best t= 200
finished: t= 303 loss= 115.18427276611328 loss val= 119.71116638183594 best val loss= 118.57266235351562 best t= 301
finished: t= 404 loss= 107.9153060913086 loss val= 111.29252624511719 best val loss= 111.29252624511719 best t= 404
finished: t= 505 loss= 105.21318817138672 loss val= 109.7998046875 best val loss= 108.2189712524414 best t= 504
finished: t= 606 loss= 102.8196792602539 loss val= 108.30864715576172 best val loss= 107.19355773925781 best t= 568
finished: t= 707 loss= 99.6415023803711 loss val= 106.8205337524414 best val loss= 106.656494140625 best t= 668
finished: t= 808 loss= 104.96278381347656 loss val= 111.1

In [7]:
ret = sorted(ret)

print(np.array(ret)[len(ret)//2], np.array(ret).std(), np.array(ret).max(), np.array(ret).min())
print(np.array(time_spend).mean(), np.array(time_spend).std())

6.004021167755127 0.9791433056681801 8.044082641601562 5.323068618774414
622.1461952030659 269.1996319054635
